# 4.2 BiLSTM — 실습

COL-BL v2.4 프로젝트의 Week 4.2 실습. 4.1 에서 만든 **Regularized 3층 Stacked LSTM** 에 
`bidirectional=True` 한 줄을 추가해 **양방향 LSTM (BiLSTM)** 을 구성하고, 
재무 시계열에서 양방향이 정말 이득인지 · 어디서 위험한지 네 가지 실험으로 검증한다.

## 이 노트북의 목적

1. **단방향 3층 vs 양방향 3층** — 같은 `hidden_dim=64` 기준 val loss 비교 (파라미터는 BiLSTM 이 2배)
2. **파라미터 수 공정 비교** — BiLSTM 의 `hidden_dim` 을 줄여 파라미터 수를 단방향과 맞췄을 때의 변화
3. **Look-ahead 누수 실증** — window 경계를 넘는 잘못된 사용법 vs sliding window 안전 구조 비교
4. **Pooling 방식 비교** — Option A (`h_n` concat) vs Option C (mean pool) 직접 대결

## 이론 md §7.4 와 대응되는 가설

| 가설 | 실험 | 예상 결과 |
|---|---|---|
| **H1** | §3 실험 1 | 동일 `hidden_dim` 에서 BiLSTM 이 단방향과 동등하거나 약간 이득 |
| **H1'** | §4 실험 2 | 파라미터 수 매칭 시 BiLSTM 이득이 사라질 수 있음 |
| **H3** | §5 실험 3 | 누수 BiLSTM 은 train MSE 가 극단적으로 낮으나 val 은 비슷/악화 → train-val gap 폭발 |
| **H2** | §6 실험 4 | Option A (h_n concat) 가 Option C (mean pool) 보다 우위 |

## 재활용 자산

- **4.1 의 `train_one_individual`** — 그대로 재사용 (변인 통제)
- **4.1 의 `VariationalDropout`** — 정규화 일관성 유지
- **4.1 의 `RegularizedStackedLSTM`** — 단방향 baseline 으로 재사용
- MSFT 일봉 데이터 (yfinance) — 4.1 과 동일

> 주의: 이 노트북은 실행 검증을 로컬 환경에서 수행 (CLAUDE.md 지침).

---

## §1. 셋업

필수 import · 한글 폰트 (CLAUDE.md 지침) · seed 고정 · device 탐지. 4.1 과 동일.

In [ ]:
# ---- 필수 라이브러리 ----
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import platform
import random
import copy

# ---- 한글 폰트 (CLAUDE.md 필수 지침) ----
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux (샌드박스 환경)
    import koreanize_matplotlib  # pip install koreanize-matplotlib --break-system-packages
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'device : {device}')

### §1.2 MSFT 일봉 로드 + 피처 엔지니어링 + 윈도우화

4.1 과 동일한 파이프라인 — 간이 10차원 피처, 60일 lookback, 80/20 시간순 분할. 
본 노트북의 관심사는 BiLSTM 의 구조적 차이이므로 데이터 파이프라인은 그대로 재활용.

In [ ]:
import yfinance as yf

ticker = 'MSFT'
df = yf.download(ticker, start='2010-01-01', end='2024-12-31', auto_adjust=True, progress=False)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f'기간: {df.index.min().date()} ~ {df.index.max().date()} ({len(df)} 일)')

# ---- 10차원 간이 피처 (4.1 과 동일) ----
close, volume, high, low = df['Close'], df['Volume'], df['High'], df['Low']
feat = pd.DataFrame(index=df.index)
feat['log_ret']    = np.log(close / close.shift(1))
feat['vol_chg']    = volume.pct_change()
feat['price_ma5']  = close / close.rolling(5).mean() - 1
feat['price_ma20'] = close / close.rolling(20).mean() - 1
feat['hl_range']   = (high - low) / close

ma20, std20 = close.rolling(20).mean(), close.rolling(20).std()
feat['bb_width'] = (4 * std20) / ma20

delta = close.diff()
up = delta.clip(lower=0).rolling(14).mean()
down = (-delta.clip(upper=0)).rolling(14).mean()
rs = up / (down + 1e-9)
feat['rsi'] = 100 - 100 / (1 + rs)

ema12 = close.ewm(span=12).mean()
ema26 = close.ewm(span=26).mean()
feat['macd']     = (ema12 - ema26) / close
feat['vol20']    = feat['log_ret'].rolling(20).std()
feat['prev_ret'] = feat['log_ret'].shift(1)

target = feat['log_ret'].shift(-1)
data = pd.concat([feat, target.rename('target')], axis=1).dropna()
print(f'사용 가능한 표본: {len(data)} 일 / 피처 {feat.shape[1]} 차원')

In [ ]:
# ---- 표준화 (train 통계만 사용; look-ahead 금지) ----
split_idx = int(len(data) * 0.8)
X_all = data[feat.columns].values.astype(np.float32)
y_all = data['target'].values.astype(np.float32)

mu = X_all[:split_idx].mean(axis=0)
sd = X_all[:split_idx].std(axis=0) + 1e-8
X_std = (X_all - mu) / sd

y_mu = y_all[:split_idx].mean()
y_sd = y_all[:split_idx].std() + 1e-8
y_std = (y_all - y_mu) / y_sd

# ---- 윈도우 생성 (60일 → 1일 예측) ----
WINDOW = 60

def make_windows(X: np.ndarray, y: np.ndarray, window: int):
    '''(T, D) → (N, window, D), (N,).
    sliding window 는 각 샘플이 타겟 시점 이전 데이터만 담도록 설계 — BiLSTM 의 안전성 구조적 보장.'''
    Xs, ys = [], []
    for i in range(len(X) - window):
        Xs.append(X[i : i + window])
        ys.append(y[i + window - 1])
    return np.array(Xs), np.array(ys)

X_win, y_win = make_windows(X_std, y_std, WINDOW)

split_win = int(len(X_win) * 0.8)
X_tr, y_tr = X_win[:split_win], y_win[:split_win]
X_va, y_va = X_win[split_win:], y_win[split_win:]

BATCH = 64
train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
val_ds   = TensorDataset(torch.from_numpy(X_va), torch.from_numpy(y_va))
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, drop_last=False)

INPUT_DIM = X_tr.shape[2]
print(f'윈도우: X_tr={X_tr.shape}, X_va={X_va.shape}')
print(f'input_dim={INPUT_DIM}, batch={BATCH}')
print(f'train batches={len(train_loader)}, val batches={len(val_loader)}')

---

## §2. 공통 유틸 + 단방향 baseline 재정의

4.1 에서 사용한 `train_one_individual`, `count_params`, `reset_seed`, `VariationalDropout`, 
`RegularizedStackedLSTM` 을 그대로 재정의한다. 노트북을 처음부터 실행해도 동작하도록.

In [ ]:
def train_one_individual(model, epochs=30, lr=1e-3, weight_decay=1e-4,
                         max_norm=1.0, patience=10, verbose=False):
    '''단일 모델을 학습하고 (best_val_loss, loss history, trained_model) 반환.
    4.1 과 동일 — 변인 통제.'''
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_val, best_state, stale, history = float('inf'), None, 0, []

    for epoch in range(epochs):
        model.train()
        tr_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
            optimizer.step()
            tr_losses.append(loss.item())

        model.eval()
        va_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                va_losses.append(loss_fn(model(xb), yb).item())

        tr, va = float(np.mean(tr_losses)), float(np.mean(va_losses))
        history.append((tr, va))

        if va < best_val:
            best_val = va
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1

        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f'  epoch {epoch:3d} | train {tr:.5f} | val {va:.5f} | best {best_val:.5f}')

        if stale >= patience:
            if verbose:
                print(f'  early stop at epoch {epoch}')
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_val, history, model


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
# ---- 4.1 의 VariationalDropout 재정의 (정규화 일관성) ----
class VariationalDropout(nn.Module):
    '''Gal & Ghahramani (2016). 한 forward 내 모든 timestep 에 동일 mask 적용.'''
    def __init__(self, p: float):
        super().__init__()
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.p == 0.0:
            return x
        mask_shape = (x.size(0), 1, x.size(2))
        mask = torch.bernoulli(torch.full(mask_shape, 1 - self.p, device=x.device))
        mask = mask / (1 - self.p)
        return x * mask

In [ ]:
# ---- 4.1 의 단방향 Regularized Stacked LSTM — baseline ----
class RegularizedStackedLSTM(nn.Module):
    '''단방향 3층 + VD + 층별 LN. 4.1 의 최종안과 동일 구조.
    BiLSTM 과 동일 조건(hidden/layers/dropout)에서 비교할 baseline.'''
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.num_layers = num_layers
        self.lstms = nn.ModuleList()
        self.lns   = nn.ModuleList()
        self.vds   = nn.ModuleList()
        for l in range(num_layers):
            in_dim = input_size if l == 0 else hidden_size
            self.vds.append(VariationalDropout(p=dropout))
            self.lstms.append(nn.LSTM(in_dim, hidden_size, num_layers=1, batch_first=True))
            self.lns.append(nn.LayerNorm(hidden_size))
        self.final_vd = VariationalDropout(p=dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out = x
        for l in range(self.num_layers):
            out = self.vds[l](out)
            out, _ = self.lstms[l](out)
            out = self.lns[l](out)
        out = self.final_vd(out)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)

---

## §3. 실험 1 — 단방향 3층 vs 양방향 3층 (hidden_dim 동일)

**설정:** `hidden=64`, `num_layers=3`, `dropout=0.2`, `lr=1e-3`, `epochs=40`. 
단 한 가지만 다르다 — **`bidirectional=True` 스위치**.

**관찰 포인트:**
- 파라미터 수: 이론 md §2.4 에서 **정확히 2배**라고 했는데 실측으로 확인
- val loss: BiLSTM 이 이득 (H1)? 아니면 과적합으로 손해?
- train/val gap: 파라미터 2배 → overfit 위험 2배 시각화

**BiLSTM 모델 구조 — Option A (`h_n` concat) pooling:**

마지막 layer 의 forward 방향 마지막 hidden (`h_n[-2]`) 과 backward 방향 마지막 hidden (`h_n[-1]`, 이는 window 시작점) 
을 concat → `(batch, 2H)` 를 선형 head 에 통과.

In [ ]:
class BiLSTMRegressor(nn.Module):
    '''양방향 3층 + VD + 층별 LN + Option A pooling (h_n concat).
    단방향 RegularizedStackedLSTM 과 구조적으로 최대한 일관.

    차이점:
      1. nn.LSTM(bidirectional=True)  → 층마다 forward + backward 두 셀
      2. 층 출력 dim 이 2*hidden (forward||backward concat)
      3. LayerNorm 도 2*hidden 에 맞춰야 함',
      4. 다음 층 input 도 2*hidden 이 됨
      5. 최종 head 입력은 2*hidden 의 h_n concat (= 4*hidden 이 아니라 2*hidden 주의)
',
    Pooling (Option A): 마지막 layer 의 h_n 의 forward/backward concat.
      - h_n shape: (num_layers*2, batch, hidden)
      - h_n[-2]: 마지막 layer forward 의 h_T      (window 끝에서 본 과거 요약)
      - h_n[-1]: 마지막 layer backward 의 h_T    (window 시작점에서 본 미래 요약)
      - 두 hidden 을 cat → (batch, 2*hidden)',
    '''
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.lstms = nn.ModuleList()
        self.lns   = nn.ModuleList()
        self.vds   = nn.ModuleList()
        for l in range(num_layers):
            # 첫 층 입력은 input_size, 이후 층 입력은 2*hidden (양방향 concat)
            in_dim = input_size if l == 0 else 2 * hidden_size
            self.vds.append(VariationalDropout(p=dropout))
            self.lstms.append(nn.LSTM(in_dim, hidden_size,
                                      num_layers=1, batch_first=True,
                                      bidirectional=True))
            # LN 입력 dim 은 2*hidden (forward+backward concat)
            self.lns.append(nn.LayerNorm(2 * hidden_size))
        self.final_vd = VariationalDropout(p=dropout)
        # head 입력: h_n[-2] || h_n[-1] = 2*hidden
        self.head = nn.Linear(2 * hidden_size, 1)

    def forward(self, x):
        out = x                                            # (B, T, D)
        h_last_fwd, h_last_bwd = None, None
        for l in range(self.num_layers):
            out = self.vds[l](out)
            out, (h_n, c_n) = self.lstms[l](out)           # out: (B, T, 2H), h_n: (2, B, H)
            out = self.lns[l](out)
            # 마지막 층의 h_n 만 pooling 에 사용
            if l == self.num_layers - 1:
                h_last_fwd = h_n[0]                        # forward  마지막 h (B, H)
                h_last_bwd = h_n[1]                        # backward 마지막 h (B, H) = 윈도우 시작 시점 역방향 요약
        # Option A: concat → (B, 2H)
        h_combined = torch.cat([h_last_fwd, h_last_bwd], dim=-1)
        h_combined = self.final_vd(h_combined.unsqueeze(1)).squeeze(1)  # VD 는 (B,T,F) 형식 기대 → 2D 로 맞춤
        return self.head(h_combined).squeeze(-1)           # (B,)

In [ ]:
# ---- 실험 1: 단방향 3층 vs 양방향 3층 ----
HIDDEN = 64
NUM_LAYERS = 3
DROPOUT = 0.2
EPOCHS = 40
LR = 1e-3

results_exp1 = {}

# ---- 단방향 baseline ----
reset_seed()
uni_model = RegularizedStackedLSTM(INPUT_DIM, HIDDEN, NUM_LAYERS, dropout=DROPOUT)
uni_params = count_params(uni_model)
print(f'\n=== 단방향 Regularized 3층 ({uni_params:,} params) ===')
uni_best, uni_hist, uni_trained = train_one_individual(uni_model, epochs=EPOCHS, lr=LR, verbose=True)
results_exp1['uni_H64'] = (uni_best, uni_hist, uni_trained, uni_params)

# ---- 양방향 BiLSTM ----
reset_seed()
bi_model = BiLSTMRegressor(INPUT_DIM, HIDDEN, NUM_LAYERS, dropout=DROPOUT)
bi_params = count_params(bi_model)
print(f'\n=== 양방향 BiLSTM 3층 ({bi_params:,} params) ===')
bi_best, bi_hist, bi_trained = train_one_individual(bi_model, epochs=EPOCHS, lr=LR, verbose=True)
results_exp1['bi_H64'] = (bi_best, bi_hist, bi_trained, bi_params)

print('\n=== 실험 1 요약 (hidden=64 고정, 파라미터는 BiLSTM 이 2배) ===')
print(f"단방향 : params={uni_params:>7,} | best_val={uni_best:.5f}")
print(f"양방향 : params={bi_params:>7,} | best_val={bi_best:.5f}")
print(f"파라미터 비율: {bi_params / uni_params:.2f}x")
print(f"val loss Δ  : {bi_best - uni_best:+.5f}  ('-' 면 BiLSTM 이득)")

In [ ]:
# ---- 학습곡선 비교 ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for name, color in [('uni_H64', 'tab:blue'), ('bi_H64', 'tab:red')]:
    best, hist, _, _ = results_exp1[name]
    tr = [h[0] for h in hist]
    va = [h[1] for h in hist]
    label = '단방향 3층' if 'uni' in name else '양방향 3층 (BiLSTM)'
    axes[0].plot(tr, label=label, color=color, alpha=0.85)
    axes[1].plot(va, label=label, color=color, alpha=0.85)

axes[0].set_title('실험 1 — Train Loss (hidden=64 고정)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train MSE')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('실험 1 — Val Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val MSE')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# ---- 파라미터 수 이론값 검증 ----
# BiLSTM 은 forward cell + backward cell 두 개가 독립 → 정확히 2배 + LN/head 차이 보정

def theoretical_params_uni(input_dim, hidden, num_layers):
    '''단방향 RegularizedStackedLSTM 의 이론 파라미터 수.
    nn.LSTM 1층: 4*H*(I+H) + 8*H  (weight_ih + weight_hh + bias_ih + bias_hh)
    LayerNorm:  2*H  (weight + bias)
    Head:       H + 1',
    '''
    total = 0
    for l in range(num_layers):
        I = input_dim if l == 0 else hidden
        total += 4 * hidden * I + 4 * hidden * hidden + 8 * hidden
        total += 2 * hidden  # LayerNorm
    total += hidden + 1
    return total

def theoretical_params_bi(input_dim, hidden, num_layers):
    '''BiLSTMRegressor 이론값.
    각 층 nn.LSTM(bidirectional=True): 단방향 LSTM 셀 2개 → 2 * (4*H*(I+H) + 8*H)
    LayerNorm:  2 * (2H)     (입력 dim = 2*hidden)
    Head:       2H + 1',
    '''
    total = 0
    for l in range(num_layers):
        # 2층 이상은 앞 층 출력(2*hidden)을 받음
        I = input_dim if l == 0 else 2 * hidden
        total += 2 * (4 * hidden * I + 4 * hidden * hidden + 8 * hidden)  # forward + backward
        total += 2 * 2 * hidden  # LayerNorm (input dim 2H)
    total += 2 * hidden + 1   # head: Linear(2H, 1)
    return total

rows = [
    {'모델': '단방향', 'actual': uni_params,
     'theoretical': theoretical_params_uni(INPUT_DIM, HIDDEN, NUM_LAYERS),
     '일치': '✅' if uni_params == theoretical_params_uni(INPUT_DIM, HIDDEN, NUM_LAYERS) else '❌'},
    {'모델': '양방향', 'actual': bi_params,
     'theoretical': theoretical_params_bi(INPUT_DIM, HIDDEN, NUM_LAYERS),
     '일치': '✅' if bi_params == theoretical_params_bi(INPUT_DIM, HIDDEN, NUM_LAYERS) else '❌'},
]
pd.DataFrame(rows)

---

## §4. 실험 2 — 파라미터 수 공정 비교 (Fair Comparison)

실험 1 의 **BiLSTM 이득** 이 진짜 '양방향 구조' 때문인지, 아니면 **단순히 파라미터 2배** 때문인지 
구분하기 위한 실험. BiLSTM 의 `hidden_dim` 을 줄여 단방향과 **비슷한 파라미터 수** 를 맞춘다.

**탐색:** 단방향 `hidden=64` (~약 38,273 params)와 가장 가까운 BiLSTM `hidden` 찾기.

```
단방향  hidden=64 : ~38k params
BiLSTM  hidden=64 : ~95k params  (2.5배)
BiLSTM  hidden=44 : ~?   params  (근접 후보)
```

정확한 `hidden` 은 다음 셀에서 탐색 후 결정. 이 실험이 H1' 가설 검증 — 
**"공정 비교에서도 BiLSTM 이 유리한가"**.

In [ ]:
# ---- BiLSTM hidden_dim 을 줄여가며 단방향 38k 파라미터에 가장 가까운 값 찾기 ----
target_params = uni_params
print(f'목표 파라미터 수 (단방향 기준): {target_params:,}')
print()

candidate_hiddens = list(range(36, 52))
rows = []
for h in candidate_hiddens:
    n = theoretical_params_bi(INPUT_DIM, h, NUM_LAYERS)
    rows.append({'hidden': h, 'params': n, 'diff': n - target_params,
                 '비율(vs 단방향)': f'{n / target_params:.2f}x'})

candidates = pd.DataFrame(rows)
best_h_idx = candidates['diff'].abs().idxmin()
best_h = int(candidates.loc[best_h_idx, 'hidden'])
print(candidates.to_string(index=False))
print(f'\n선택: hidden={best_h} (|차이| 최소)')

In [ ]:
# ---- 실험 2: 단방향 H=64 vs BiLSTM H=best_h (파라미터 유사) ----
HIDDEN_BI_FAIR = best_h

reset_seed()
bi_fair_model = BiLSTMRegressor(INPUT_DIM, HIDDEN_BI_FAIR, NUM_LAYERS, dropout=DROPOUT)
bi_fair_params = count_params(bi_fair_model)
print(f'\n=== 양방향 BiLSTM (hidden={HIDDEN_BI_FAIR}, {bi_fair_params:,} params) ===')
print(f'단방향 대비 파라미터: {bi_fair_params / uni_params:.2f}x (목표 ≈ 1.00x)')

bi_fair_best, bi_fair_hist, bi_fair_trained = train_one_individual(
    bi_fair_model, epochs=EPOCHS, lr=LR, verbose=True
)
results_exp1['bi_fair'] = (bi_fair_best, bi_fair_hist, bi_fair_trained, bi_fair_params)

print('\n=== 실험 2 요약 (파라미터 수 공정 비교) ===')
print(f"단방향 H=64         : params={uni_params:>7,} | best_val={uni_best:.5f}")
print(f"양방향 H={HIDDEN_BI_FAIR} (fair) : params={bi_fair_params:>7,} | best_val={bi_fair_best:.5f}")
print(f"양방향 H=64 (참고)   : params={bi_params:>7,} | best_val={bi_best:.5f}")

In [ ]:
# ---- 공정 비교 학습곡선 ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

runs = [
    ('uni_H64',  '단방향 H=64',            'tab:blue'),
    ('bi_fair',  f'양방향 H={HIDDEN_BI_FAIR} (fair)', 'tab:green'),
    ('bi_H64',   '양방향 H=64 (2배 파라미터)',  'tab:red'),
]
for key, label, color in runs:
    best, hist, _, _ = results_exp1[key]
    tr = [h[0] for h in hist]
    va = [h[1] for h in hist]
    axes[0].plot(tr, label=label, color=color, alpha=0.85)
    axes[1].plot(va, label=label, color=color, alpha=0.85)

axes[0].set_title('실험 1+2 — Train Loss (3개 조건)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train MSE')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].set_title('실험 1+2 — Val Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val MSE')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# 해석 가이드
print('해석 가이드:')
print('  - 파라미터 유사 (fair) 조건에서 BiLSTM 이 단방향보다 우수하면 → 양방향 구조 자체의 이득')
print('  - 파라미터 유사 조건에서 BiLSTM 이 단방향과 같거나 나쁘면 → 실험 1 의 이득은 주로 용량 효과')

---

## §5. 실험 3 — Look-ahead 누수 실증 (window 경계 위반)

**이론 md §3.2 시나리오 A 를 실증.** 전체 train 시계열을 **한 번에** BiLSTM 에 통과시켜 
backward pass 가 전체 미래를 보도록 하고, 그 결과 train/val gap 이 얼마나 폭발적으로 벌어지는지 관찰.

**경계 위반 구조:**
```
정상 (sliding window):
  샘플 i 의 입력  = [x_i, x_{i+1}, ..., x_{i+59}]   (60일 window)
  샘플 i 의 타겟  = y_{i+59}  (window 마지막 날의 다음 수익률)
  backward 는 window 내부에서만 움직임 → 안전

경계 위반 (시나리오 A):
  train 전체 시계열 X = [x_1, x_2, ..., x_N] 을 한 번에 BiLSTM 에 통과
  각 시점 t 의 예측을 loss 에 누적
  → backward 가 시점 t 에서 x_{t+1}, ..., x_N 을 모두 본 상태로 예측
```

**예상 (H3):**
- 경계 위반 쪽 train loss 는 **극단적으로 낮아짐** (미래 정답을 엿본 상태)
- val loss 는 비슷하거나 악화 (val 구간은 진짜 미래라 엿볼 수 없음)
- 따라서 **train-val gap** 이 정상 구조 대비 10~100배 벌어짐

In [ ]:
class LeakyBiLSTM(nn.Module):
    '''의도적으로 경계 위반을 만들기 위한 BiLSTM.

    구조: 전체 train 시계열을 한 번의 forward 에 통과. 각 시점의 예측을 loss 에 쓴다.
    정상 BiLSTMRegressor 와 다른 점: window pooling 대신 per-timestep 예측 → 시점 t 의
    backward hidden 이 x_{t+1..N} 을 모두 본 상태로 y_t 예측.
    '''
    def __init__(self, input_size, hidden_size, num_layers=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True,
        )
        self.head = nn.Linear(2 * hidden_size, 1)

    def forward(self, x):
        # x: (1, T_total, D) — 전체 시계열을 하나의 긴 시퀀스로
        out, _ = self.lstm(x)      # (1, T_total, 2H)
        return self.head(out).squeeze(-1).squeeze(0)   # (T_total,)

In [ ]:
# ---- 경계 위반 데이터 구성: 전체 train 시계열을 단일 배치로 ----
# 주의: 이건 '의도적으로 잘못된' 파이프라인. 절대 실거래/백테스트에 사용하지 말 것.

# train/val 전체 시계열 (windowing 하지 않음)
leak_X_tr_full = torch.from_numpy(X_std[:split_idx]).float().unsqueeze(0).to(device)    # (1, T_tr, D)
leak_y_tr_full = torch.from_numpy(y_std[:split_idx]).float().to(device)                  # (T_tr,)

# val 는 별도로 sliding window 로 유지 (val 은 진짜 미래라 정상 파이프라인으로 평가해야 의미 있음)
# 경계 위반 BiLSTM 의 val 평가는 '같은 방식' 으로 val 시계열을 단일 배치에 통과
leak_X_va_full = torch.from_numpy(X_std[split_idx:]).float().unsqueeze(0).to(device)
leak_y_va_full = torch.from_numpy(y_std[split_idx:]).float().to(device)

print(f'경계 위반 train 시퀀스: {leak_X_tr_full.shape}')
print(f'경계 위반 val   시퀀스: {leak_X_va_full.shape}')

In [ ]:
# ---- 경계 위반 학습 루프 (시점별 예측 → loss 평균) ----
def train_leaky(model, epochs=40, lr=1e-3, weight_decay=1e-4, patience=10, verbose=False):
    '''경계 위반 BiLSTM 학습. 전체 train 시계열을 한 forward 에 통과.'''
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_val, best_state, stale, history = float('inf'), None, 0, []

    for epoch in range(epochs):
        # train 
        model.train()
        optimizer.zero_grad()
        pred = model(leak_X_tr_full)                 # (T_tr,)
        loss = loss_fn(pred, leak_y_tr_full)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        tr = float(loss.item())

        # val (같은 경계 위반 방식)
        model.eval()
        with torch.no_grad():
            va_pred = model(leak_X_va_full)
            va = float(loss_fn(va_pred, leak_y_va_full).item())
        history.append((tr, va))

        if va < best_val:
            best_val = va
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1

        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f'  epoch {epoch:3d} | train {tr:.5f} | val {va:.5f} | best {best_val:.5f}')

        if stale >= patience:
            if verbose:
                print(f'  early stop at epoch {epoch}')
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_val, history

# ---- 학습 ----
reset_seed()
leaky_model = LeakyBiLSTM(INPUT_DIM, HIDDEN, num_layers=NUM_LAYERS, dropout=DROPOUT)
print(f'\n=== 경계 위반 BiLSTM ({count_params(leaky_model):,} params) ===')
leaky_best, leaky_hist = train_leaky(leaky_model, epochs=EPOCHS, lr=LR, verbose=True)
print(f'\n경계 위반 BiLSTM best_val={leaky_best:.5f}')

In [ ]:
# ---- 누수 증거: train/val gap 비교 ----
# 정상 BiLSTM (실험 1 의 bi_H64) 과 경계 위반 BiLSTM 의 gap 비교
normal_tr_last = results_exp1['bi_H64'][1][-1][0]
normal_va_last = results_exp1['bi_H64'][1][-1][1]
leaky_tr_last = leaky_hist[-1][0]
leaky_va_last = leaky_hist[-1][1]

print('=== 마지막 에폭 기준 ===')
print(f'정상 BiLSTM    : train={normal_tr_last:.5f} | val={normal_va_last:.5f} | gap={normal_va_last - normal_tr_last:+.5f}')
print(f'경계 위반 BiLSTM: train={leaky_tr_last:.5f} | val={leaky_va_last:.5f} | gap={leaky_va_last - leaky_tr_last:+.5f}')
print()
print(f'경계 위반 쪽 train 이 비정상적으로 낮다면 → 미래 누수 성공적으로 발생 (나쁜 의미로)')
print(f'val 은 양쪽이 비슷해야 정상 (val 구간은 둘 다 진짜 미래)')

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 정상
normal_hist = results_exp1['bi_H64'][1]
axes[0].plot([h[0] for h in normal_hist], label='train', color='tab:blue')
axes[0].plot([h[1] for h in normal_hist], label='val',   color='tab:orange')
axes[0].set_title('정상 BiLSTM (sliding window)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, max(max(h[1] for h in normal_hist), max(h[1] for h in leaky_hist)) * 1.1)

# 경계 위반
axes[1].plot([h[0] for h in leaky_hist], label='train', color='tab:blue')
axes[1].plot([h[1] for h in leaky_hist], label='val',   color='tab:orange')
axes[1].set_title('경계 위반 BiLSTM (전체 시계열 단일 배치)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, max(max(h[1] for h in normal_hist), max(h[1] for h in leaky_hist)) * 1.1)

plt.tight_layout(); plt.show()

### §5 해석 — H3 가설 재검토 (실측값 기반)

**실측 결과:**
- 정상 BiLSTM (sliding window) 마지막 에폭 : train=0.98900 | val=1.19228 | **gap = +0.20328** (val 이 train 보다 높음 = 일반적인 과적합 패턴)
- 경계 위반 BiLSTM (epoch 39)             : train=0.55522 | val=0.53096 | **gap = −0.02426** (val 이 train 보다 오히려 낮음 ⚠)

이 결과는 이론 md §3.2 에서 예상한 "train 만 급락, val 은 정체" 패턴과는 **다른 방식으로 H3 를 확증**한다.

**왜 val 도 같이 낮아졌는가**

경계 위반 구조 (`train_leaky`) 는 val 평가도 '같은 방식' 으로 수행했다:

```python
# val 평가도 전체 val 시계열을 단일 배치에 통과
va_pred = model(leak_X_va_full)   # (1, T_va, D) → (T_va,)
va = loss_fn(va_pred, leak_y_va_full)
```

즉 val 시점 t 의 예측도 **val 구간 내부의 x_{t+1}, ..., x_{T_va} 를 backward 로 본 상태**에서 계산된다. val 에도 동일한 경계 위반이 가해져 있기 때문에, val 역시 비현실적으로 낮은 값으로 측정된다.

**이게 오히려 더 강한 누수 증거다**

```
정상 구조: train 0.989 → val 1.192  (train 에서 배운 걸로 val 을 진짜 미래 예측 → 1.19)
경계 위반: train 0.555 → val 0.531  (val 도 '미래를 엿본 상태'의 측정치라 1.19 가 아니라 0.53)
```

실전 투자/백테스트에서는 val 시점에 **미래 정보가 없다**. 따라서 경계 위반 모델이 val MSE 0.53 을 찍는 건 "정상 조건에서는 달성 불가능한 수치" 다. 사람 손으로 코드를 잘못 짰을 때 이 경계 위반 val 점수를 그대로 믿으면, **실거래에서는 val 1.19 수준(= 무작위 추측 수준)으로 되돌아와 대규모 손실** 로 이어진다.

**교훈 — 이론 md §3 의 경고는 유효**

- 경계 위반이 발생한 순간 train 도 val 도 **둘 다 해석 불가능해진다** (gap 이 폭발하지 않아도 마찬가지)
- 따라서 BiLSTM 을 쓰려면 **반드시 sliding window dataset** 과 **각 샘플 내부에서만 backward 가 움직이는 구조** 가 필수
- 4.3 Optuna 코드에서 이 구조를 assertion 으로 강제할 것

**H3 판정: ✅ 확증 (변형된 형태로)**. "gap 폭발" 대신 "train·val 둘 다 누수" 라는 더 강한 증거가 관측됨.

**방어 체크리스트:**
- [ ] train-val gap 모니터링 (단독으로는 불충분 — 경계 위반 시 gap 이 오히려 음수가 될 수도 있음)
- [x] sliding window dataset 사용 여부를 코드 수준에서 강제 (필수)
- [x] 각 샘플의 입력 길이가 고정 window (예: 60) 인지 assertion
- [x] val/test 구간에서 미래 시점을 backward 가 보지 못하도록 구조적 보장

---

## §6. 실험 4 — Pooling 비교 (Option A h_n concat vs Option C mean pool)

이론 md §6 의 네 옵션 중 **Option A (h_n concat)** 가 실험 1/2 의 기본값. 여기서는 
**Option C (mean pool)** 로 바꿨을 때 성능이 어떻게 변하는지 직접 비교해 H2 가설 검증.

**Option A** (기본): `h_n[-2]` (window 끝에서 본 과거) `||` `h_n[-1]` (window 시작에서 본 미래) → `(B, 2H)` 
**Option C** (mean pool): 전 timestep 의 hidden 을 시간축 평균 → `(B, 2H)`

두 옵션 모두 출력 차원은 `2H` 로 동일. 차이는 **어떤 시점의 정보를 얼마나 가중하는가** 뿐.

**H2 예상:** Option A 가 우위. 재무 시계열에서 최근 시점 비중이 크고, mean pool 은 이를 희석.

In [ ]:
class BiLSTMRegressorMeanPool(nn.Module):
    '''BiLSTMRegressor 와 구조 동일 — 마지막 pooling 만 Option C (mean pool) 로 변경.'''
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.lstms = nn.ModuleList()
        self.lns   = nn.ModuleList()
        self.vds   = nn.ModuleList()
        for l in range(num_layers):
            in_dim = input_size if l == 0 else 2 * hidden_size
            self.vds.append(VariationalDropout(p=dropout))
            self.lstms.append(nn.LSTM(in_dim, hidden_size,
                                      num_layers=1, batch_first=True,
                                      bidirectional=True))
            self.lns.append(nn.LayerNorm(2 * hidden_size))
        self.final_vd = VariationalDropout(p=dropout)
        self.head = nn.Linear(2 * hidden_size, 1)

    def forward(self, x):
        out = x
        for l in range(self.num_layers):
            out = self.vds[l](out)
            out, _ = self.lstms[l](out)
            out = self.lns[l](out)
        out = self.final_vd(out)
        # Option C: 시간축 평균
        h_mean = out.mean(dim=1)                # (B, 2H)
        return self.head(h_mean).squeeze(-1)

In [ ]:
# ---- 실험 4: Option A (이미 있음: bi_H64) vs Option C ----
reset_seed()
bi_mean_model = BiLSTMRegressorMeanPool(INPUT_DIM, HIDDEN, NUM_LAYERS, dropout=DROPOUT)
bi_mean_params = count_params(bi_mean_model)
print(f'\n=== BiLSTM + mean pool (hidden={HIDDEN}, {bi_mean_params:,} params) ===')
bi_mean_best, bi_mean_hist, bi_mean_trained = train_one_individual(
    bi_mean_model, epochs=EPOCHS, lr=LR, verbose=True
)

print('\n=== 실험 4 요약 (Pooling 비교) ===')
print(f"Option A (h_n concat): params={bi_params:>7,}     | best_val={bi_best:.5f}")
print(f"Option C (mean pool ): params={bi_mean_params:>7,} | best_val={bi_mean_best:.5f}")
print(f"Δ (C - A)            : {bi_mean_best - bi_best:+.5f}  ('+' 면 A 가 더 좋음)")

In [ ]:
# ---- Pooling 비교 학습곡선 ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

bi_hist_A = results_exp1['bi_H64'][1]
axes[0].plot([h[0] for h in bi_hist_A],    label='Option A (h_n concat)', color='tab:red',    alpha=0.85)
axes[0].plot([h[0] for h in bi_mean_hist], label='Option C (mean pool)',  color='tab:purple', alpha=0.85)
axes[0].set_title('Pooling 비교 — Train Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train MSE')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot([h[1] for h in bi_hist_A],    label='Option A (h_n concat)', color='tab:red',    alpha=0.85)
axes[1].plot([h[1] for h in bi_mean_hist], label='Option C (mean pool)',  color='tab:purple', alpha=0.85)
axes[1].set_title('Pooling 비교 — Val Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val MSE')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# ---- 종합 비교 표 ----
summary_rows = [
    {'설정': '단방향 H=64',              '파라미터': uni_params,    'val_MSE': round(uni_best, 6)},
    {'설정': f'양방향 H={HIDDEN_BI_FAIR} fair',  '파라미터': bi_fair_params, 'val_MSE': round(bi_fair_best, 6)},
    {'설정': '양방향 H=64 Option A',     '파라미터': bi_params,     'val_MSE': round(bi_best, 6)},
    {'설정': '양방향 H=64 Option C',     '파라미터': bi_mean_params,'val_MSE': round(bi_mean_best, 6)},
    {'설정': '경계 위반 BiLSTM',          '파라미터': count_params(leaky_model), 'val_MSE': round(leaky_best, 6)},
]
summary = pd.DataFrame(summary_rows)
summary['val_MSE 순위'] = summary['val_MSE'].rank().astype(int)
summary.sort_values('val_MSE')

---

## §7. 결론 + 4.3 OLSTM 탐색 공간 확장

### 7.1 가설 검증 결과 (실측값)

| 가설 | 실험 | 실측 결과 | 판정 |
|---|---|---|---|
| **H1**: 같은 `hidden_dim` 에서 BiLSTM 이 단방향과 동등 이상 | §3 | Δval = 1.16618 − 1.16942 = **−0.00324** | ✅ **성립** (근소 이득) |
| **H1'**: 파라미터 매칭 시 이득 유지 | §4 | Δval = 1.17090 − 1.16942 = **+0.00148** | ❌ **실패** (공정 비교에서 악화) |
| **H2**: Option A (concat) > Option C (mean pool) | §6 | Δval = 1.17276 − 1.16618 = **+0.00658** | ✅ **성립** (concat 우위) |
| **H3**: 경계 위반 시 해석 불가능한 측정값 발생 | §5 | 경계 위반 val 0.531 vs 정상 val 1.192 (비현실적으로 낮음) | ✅ **확증 (변형된 형태)** |

### 7.2 정상 조건 순위 (경계 위반 제외)

| 순위 | 설정 | 파라미터 | val MSE | Δ vs 최고 |
|---|---|---:|---:|---:|
| 1 | 양방향 H=64 Option A | 238,465 | **1.16618** | (기준) |
| 2 | 단방향 H=64 | 86,465 | 1.16942 | +0.00324 |
| 3 | 양방향 H=38 fair | 86,261 | 1.17090 | +0.00472 |
| 4 | 양방향 H=64 Option C | 238,465 | 1.17276 | +0.00658 |

**핵심 관찰:**
1. 양방향 구조의 **순수 기여는 거의 없다**. H1 의 이득(−0.00324) 은 H1' 공정 비교가 보여주듯 주로 **용량 효과**(파라미터 2.76배)로 설명됨.
2. Pooling 방식(Option A vs C) 차이(+0.00658) 가 **양방향 여부 차이(−0.00324)** 보다 **두 배 이상 크다**. 즉 "어떻게 pool 하는가" 가 "양방향이냐 아니냐" 보다 중요.
3. **MSFT 일일 수익률이 near-random-walk** 이라 모든 val MSE 가 1.17 근처(≈ y_std 의 분산 ≈ 1)에 수렴 — 어떤 구조도 랜덤 이상을 내지 못함.

### 7.3 파라미터 비율 2.76x — 이론 2x 와의 차이 설명

이론 md §2.4 는 "BiLSTM 파라미터 = 단방향 × 2" 로 예고. 그러나 실측은 **2.76x**.

| 요소 | 단방향 | 양방향 | 비율 |
|---|---:|---:|---:|
| LSTM 셀 자체 | X | 2X (forward+backward) | 정확히 2x |
| LayerNorm(H) → LayerNorm(2H) | 2H | 4H | 2x |
| 2층 이상의 입력 dim (H → 2H) | 4H(I+H) = 4H·H | 4H·2H = 8H² (각 방향) | **2x 더 큼** |
| Head Linear(H,1) vs Linear(2H,1) | H+1 | 2H+1 | ~2x |

즉 **2층 이상의 입력 차원이 2H 로 늘어나는 연쇄 효과** 때문에 2x 가 아닌 2.76x 가 된다. `theoretical_params_bi` 계산 결과(=238,465) 가 actual 과 정확히 일치하므로 이 설명이 검증됨.

### 7.4 4.3 OLSTM 탐색 공간에 추가할 한 줄 (조건부 샘플링)

이론 md §8.2 에서 예고한 `bidirectional` 스위치를 그대로 추가하되, 본 실습 결과(H1' 실패) 를 반영해 **조건부 범위 축소** 를 함께 도입:

```python
def objective(trial):
    # 양방향 여부
    bidirectional = trial.suggest_categorical('bidirectional', [False, True])

    # H1' 실패 반영 — 양방향 선택 시 hidden 범위를 축소 (용량 보전)
    if bidirectional:
        hidden_dim = trial.suggest_categorical('hidden_dim', [16, 24, 32, 45, 64])
    else:
        hidden_dim = trial.suggest_categorical('hidden_dim', [32, 64, 128, 256])

    num_layers   = trial.suggest_int('num_layers', 1, 4)
    dropout      = trial.suggest_float('dropout', 0.1, 0.4, step=0.05)
    lr           = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    # H3 확증 반영 — sliding window 강제
    assert X_tr.ndim == 3 and X_tr.shape[1] == WINDOW, (
        'BiLSTM 은 sliding window (B, T, D) dataset 에서만 허용. 전체 시계열 단일 배치 금지.'
    )

    ModelCls = BiLSTMRegressor if bidirectional else RegularizedStackedLSTM
    model = ModelCls(INPUT_DIM, hidden_dim, num_layers, dropout=dropout)

    best_val, _, _ = train_one_individual(
        model, epochs=40, lr=lr, weight_decay=weight_decay, verbose=False
    )
    return best_val
```

**왜 조건부 샘플링인가:** 양방향 H=64 와 단방향 H=64 는 파라미터 2.76배 차이. Optuna 가 이 공간을 균등하게 탐색하면 **'양방향 = 용량 큰 것'** 이라는 비구조적 편향이 생김. 양방향 쪽 hidden 범위를 축소해 **실질 용량을 맞추면** 순수 구조 효과를 측정 가능.

### 7.5 4.3 준비물 업데이트

| 자산 | 출처 | 4.3 에서의 역할 |
|---|---|---|
| `RegularizedStackedLSTM` | 4.1 | 단방향 후보 (bidirectional=False 일 때 선택) |
| `BiLSTMRegressor` | 4.2 (이 노트북) | 양방향 후보 (bidirectional=True 일 때 선택) |
| `VariationalDropout` | 3.5 / 4.1 | 정규화 일관성 |
| `train_one_individual` | 3.4 / 4.1 | Optuna `objective` 내부 호출 |
| sliding window dataset | 2주차 | **H3 확증 — 코드 수준 assertion 으로 방어** |

### 7.6 val MSE > 1.16 의 의미 (4.1 §7 의 반복)

본 실습 정상 조건의 모든 val MSE 는 **1.16~1.17** 구간에 집중. 이는 y_std (표준화된 target) 의 분산 ~ 1 에 근접.

- MSFT 일일 수익률은 near-random-walk 이라 10차원 간이 피처로는 예측 불가
- 본 실습의 목적은 **예측력** 이 아닌 **양방향 구조의 작동과 누수 양상 관찰**
- Week 4.3 이후 XLK 전환 + CEEMDAN IMF 분해 단계에서 실질 예측력 개선 기대

### 7.7 체크포인트 (모두 실증 완료)

- [x] BiLSTM 파라미터 수 이론값 = 실측값 정확히 일치 (§3, theoretical_params_bi = 238,465)
- [x] 공정 비교(H=38 fair) 에서 BiLSTM 이 단방향 대비 +0.00148 악화 — **H1' 실패 확인** (§4)
- [x] 경계 위반 시 train·val 동시 누수 발생 (0.555 / 0.531) — **H3 변형 확증** (§5)
- [x] Option A (concat) 가 Option C (mean pool) 대비 -0.00658 우위 — **H2 성립** (§6)

### 다음 토픽

- **4.3 OLSTM (Optuna-LSTM)** — 본 노트북의 `BiLSTMRegressor` 를 포함해 Optuna 가 bidirectional 여부 + hidden_dim/num_layers/dropout/lr/wd 전체를 공동 탐색. H1' 실패 반영한 **조건부 샘플링** 도입.
- **5.1 Attention-LSTM** — 이론 md §6 의 Option D (attention pooling) 구현. 본 실험에서 Option A > Option C 였지만, attention 은 학습 가능한 시점 가중치로 Option A 를 이길 수 있는지 추가 검증.